# Liveness Model — Training & EvaluationTrains two models on subject-grouped splits and evaluates them once at a thresholdfixed on validation.- **Model A** single-frame CNN — the control- **Model B** CNN + BiLSTM — the temporal modelThen scores both against three external datasets containing **attack types never seenin training**.**Before running:** attach all four datasets via *Add Data*:`attacks-with-2d-printed-masks-of-indian-people`, `real-vs-fake-anti-spoofing-video-classification`,`cut-out-printout-attacks`, `biometric-attacks-in-different-lighting`.Set the accelerator to **GPU T4**.

## 1. Setup

In [ ]:
!pip install -q insightface onnxruntime-gpu 2>&1 | tail -2import os, sys, glob, json, shutil, subprocess# Get the project code. Either clone it, or upload the repo as a Kaggle dataset.REPO_URL = "https://github.com/madhav-sharma0201/secure-biometric-attendance.git"CODE_DIR = "/kaggle/working/secure-biometric-attendance"if REPO_URL:    if not os.path.isdir(CODE_DIR):        subprocess.run(["git", "clone", "-q", REPO_URL, CODE_DIR], check=True)else:    # fallback: repo uploaded as a dataset    cands = glob.glob("/kaggle/input/*/ml/liveness/models.py")    if not cands:        raise SystemExit(            "No code found. Either set REPO_URL above, or upload the repo as a "            "Kaggle dataset (Add Data > Upload) so that ml/ sits at its root."        )    src = os.path.dirname(os.path.dirname(os.path.dirname(cands[0])))    shutil.copytree(src, CODE_DIR, dirs_exist_ok=True)sys.path.insert(0, CODE_DIR)os.chdir(CODE_DIR)print("code at", CODE_DIR)import torchprint("torch", torch.__version__, "| cuda", torch.cuda.is_available(),      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. Locate the attached datasets

In [ ]:
INPUT = "/kaggle/input"print("attached:", os.listdir(INPUT))def find(*keywords):    """Locate an attached dataset directory by keyword, tolerating slug differences."""    for d in os.listdir(INPUT):        low = d.lower()        if all(k in low for k in keywords):            return os.path.join(INPUT, d)    return NoneDATASETS = {    "primary":        find("printed", "mask", "indian"),    "real_vs_fake":   find("real", "fake"),    "printout_masks": find("printout") or find("cut", "out"),    "lighting":       find("lighting"),}for k, v in DATASETS.items():    print(f"  {k:<16} {v}")if DATASETS["primary"] is None:    raise SystemExit("primary dataset not attached — add attacks-with-2d-printed-masks-of-indian-people")

## 3. Build manifestsOne row per clip: path, label, subject, attack type.

In [ ]:
import csvfrom scripts.build_manifest import printed_masks, trainingdatapro, printout_masks, lightingADAPT = {    "primary":        printed_masks,    "real_vs_fake":   trainingdatapro,    "printout_masks": printout_masks,    "lighting":       lighting,}os.makedirs("/kaggle/working/manifests", exist_ok=True)MANIFESTS = {}for name, root in DATASETS.items():    if root is None:        print(f"{name:<16} not attached, skipping")        continue    rows = ADAPT[name](root)    out = f"/kaggle/working/manifests/{name}.csv"    with open(out, "w", newline="") as fh:        w = csv.DictWriter(fh, fieldnames=["path","label","subject","clip","attack_type","session"])        w.writeheader(); w.writerows(rows)    MANIFESTS[name] = out    from collections import Counter    print(f"{name:<16} {len(rows):>4} clips  "          f"{len({r['subject'] for r in rows}):>3} subjects  "          f"{dict(Counter(r['label'] for r in rows))}")

## 4. Preprocess — detect, align and cache face cropsThe same detector used here is used at serving time. Crops differing between trainingand deployment would shift the input distribution and invalidate the measured metrics.

In [ ]:
from ml.preprocessing.face_processor import FaceProcessorfrom ml.preprocessing.extract import process_manifestimport timeMAX_FRAMES, STRIDE = 16, 3CROPS = {}proc = FaceProcessor(image_size=112, ctx_id=0 if torch.cuda.is_available() else -1)for name, mpath in MANIFESTS.items():    out_dir = f"/kaggle/working/crops/{name}"    if os.path.isdir(out_dir) and os.path.exists(f"{out_dir}/manifest.csv"):        print(f"{name}: cached"); CROPS[name] = out_dir; continue    print(f"\n--- {name} ---")    t0 = time.time()    process_manifest(mpath, out_dir, proc, stride=STRIDE, max_frames=MAX_FRAMES)    print(f"{name} took {(time.time()-t0)/60:.1f} min")    CROPS[name] = out_dir

## 5. Subject-grouped splitsSplit on **subject**, so the test set contains people the model has never seen.`assert_no_leakage` is a hard invariant, not a convention.

In [ ]:
from ml.preprocessing.splits import load_manifest, split_by_group, summarise_byprimary_rows = load_manifest(f"{CROPS['primary']}/manifest.csv")primary_rows = [r for r in primary_rows if int(r["n_frames"]) > 0]# manifest.csv from preprocessing keeps subject/attack_type, so grouping still workssplits = split_by_group(primary_rows, train_frac=0.6, val_frac=0.2,                        seed=42, group_key="subject")print(summarise_by(splits, group_key="subject"))print()print("test subjects:", sorted({r["subject"] for r in splits["test"]}))external = {}for name in ("real_vs_fake", "printout_masks", "lighting"):    if name in CROPS:        rows = load_manifest(f"{CROPS[name]}/manifest.csv")        external[name] = [r for r in rows if int(r["n_frames"]) > 0]        print(f"external {name}: {len(external[name])} clips")

## 6. Model A — single-frame CNN (control)

In [ ]:
import yaml, copyfrom ml.training.train import trainBASE = yaml.safe_load(open("configs/liveness.yaml"))BASE["data"]["sequence_length"] = 8cfg_a = copy.deepcopy(BASE); cfg_a["model"]["arch"] = "cnn"cfg_a["augmentation"]["enabled"] = Truelog_a = train(cfg_a, splits, CROPS["primary"], "/kaggle/working/models", "modelA_cnn")

## 7. Model B — CNN + BiLSTM (temporal)

In [ ]:
cfg_b = copy.deepcopy(BASE); cfg_b["model"]["arch"] = "cnn_lstm"cfg_b["augmentation"]["enabled"] = Truelog_b = train(cfg_b, splits, CROPS["primary"], "/kaggle/working/models", "modelB_cnnlstm")

## 8. Evaluate both — threshold fixed on validation, test touched onceExternal sets are scored at the **same** threshold. Re-tuning per set would turn theminto validation data and the "unseen" claim would be false.

In [ ]:
from ml.evaluation.evaluate import evaluate_all, format_report, save_reportfrom ml.liveness.models import build_modeldevice = "cuda" if torch.cuda.is_available() else "cpu"crops_dirs = {"primary": CROPS["primary"], **{k: CROPS[k] for k in external}}reports = {}for tag, cfg, run_id in [("Model A (CNN)", cfg_a, "modelA_cnn"),                         ("Model B (CNN-LSTM)", cfg_b, "modelB_cnnlstm")]:    ckpt = torch.load(f"/kaggle/working/models/{run_id}_best.pt", map_location=device)    model = build_model(cfg["model"]).to(device)    model.load_state_dict(ckpt["model"])    mode = "frame" if cfg["model"]["arch"] == "cnn" else "sequence"    rep = evaluate_all(model, splits, external, crops_dirs, mode,                       cfg["data"]["sequence_length"], device)    reports[run_id] = rep    save_report(rep, f"/kaggle/working/reports/{run_id}.json")    print("=" * 70); print(tag); print("=" * 70)    print(format_report(rep)); print()

## 9. Ablation — did the temporal model actually help?

In [ ]:
a, b = reports["modelA_cnn"], reports["modelB_cnnlstm"]print(f"{'model':<22}{'test ACER':>11}{'APCER':>9}{'BPCER':>9}{'AUC':>8}")for tag, r in [("A single-frame CNN", a), ("B CNN + BiLSTM", b)]:    t = r["test"]    print(f"{tag:<22}{t['acer']*100:>10.2f}%{t['apcer']*100:>8.2f}%"          f"{t['bpcer']*100:>8.2f}%{t['roc_auc']:>8.3f}")delta = (a["test"]["acer"] - b["test"]["acer"]) * 100print(f"\ntemporal model changes ACER by {delta:+.2f} percentage points")print("NOTE: with a small test set this may be within noise — check the CIs above")print(f"\n{'set':<18}{'A ACER':>10}{'B ACER':>10}")for name in a["external"]:    print(f"{name:<18}{a['external'][name]['acer']*100:>9.2f}%"          f"{b['external'][name]['acer']*100:>9.2f}%")# handheld vs static: the hypothesis for why temporal should helpprint("\nper-attack APCER (static vs hand-held masks)")print(f"{'attack':<34}{'A':>9}{'B':>9}")for k in sorted(set(a['test']['apcer_per_attack']) | set(b['test']['apcer_per_attack'])):    va = a['test']['apcer_per_attack'].get(k, float('nan')) * 100    vb = b['test']['apcer_per_attack'].get(k, float('nan')) * 100    print(f"{k:<34}{va:>8.1f}%{vb:>8.1f}%")

## 10. Export the better model to ONNX

In [ ]:
from ml.inference.export_onnx import exportbest_id = "modelB_cnnlstm" if b["test"]["acer"] <= a["test"]["acer"] else "modelA_cnn"print("exporting", best_id)onnx_path = export(f"/kaggle/working/models/{best_id}_best.pt",                   f"/kaggle/working/models/liveness.onnx",                   seq_len=BASE["data"]["sequence_length"])meta = {    "model_id": best_id,    "threshold": reports[best_id]["threshold"],    "sequence_length": BASE["data"]["sequence_length"],    "image_size": 112,    "test_acer": reports[best_id]["test"]["acer"],    "test_apcer": reports[best_id]["test"]["apcer"],    "test_bpcer": reports[best_id]["test"]["bpcer"],}json.dump(meta, open("/kaggle/working/models/liveness_meta.json", "w"), indent=2)print(json.dumps(meta, indent=2))print(f"\nsize: {os.path.getsize(onnx_path)/1e6:.1f} MB")

## 11. DownloadFrom the notebook's **Output** tab, download:- `models/liveness.onnx` and `models/liveness_meta.json` → the backend loads these- `reports/*.json` → the measured numbers for the README- `models/*_run.json` → training curvesPaste the Section 8 and 9 output back into the chat and I will wire the backendaround these exact numbers.